In [1]:
import os, sys
import pyspark

os.environ["PYSPARK_PYTHON"]=sys.executable          # Worker가 사용할 Python 실행 파일 경로 설정 (리눅스 "/usr/bin/python3")
os.environ["PYSPARK_DRIVER_PYTHON"]=sys.executable   # Driver에서도 동일한 Python 경로 설정
# os.environ['HADOOP_HOME']=os.getcwd() # 현재 디렉터리를 HADOOP_HOME으로 설정
# os.environ["PATH"] += os.path.join(os.environ['HADOOP_HOME'], 'bin') # PATH에 Hadoop 바이너리 추가

myConf=pyspark.SparkConf() # 기본 설정 객체 생성, 여기에 필요한 설정 정의
# myConf=pyspark.SparkConf().set("spark.driver.bindAddress", "127.0.0.1") #드라이버 바인딩 주소 설정
# myConf=pyspark.SparkConf().set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.1.1") 

In [2]:
spark = pyspark.sql.SparkSession\
    .builder\
    .master("local")\
    .appName("test")\
    .config(conf=myConf)\
    .getOrCreate()

In [3]:
_sub=["20150101,2호선,0236,영등포구청,6199,6219",

"20150101,2호선,0237,당산,7982,8946",

"20150101,2호선,0238,합정,17406,15241",

"20150101,3호선,0309,지축,515,538",

"20150101,3호선,0310,구파발,6879,6260",

"20150101,3호선,0311,연신내,20031,19470",

"20150101,3호선,0312,불광,9519,11029",

"20150101,4호선,0425,회현,7465,7574",

"20150101,4호선,0426,서울역,3943,10823",

"20150101,경부선,1002,남영,4340,4535",

"20150101,경부선,1003,용산,28980,27684",

"20150101,경부선,1004,노량진,23021,23862",

"20150101,경부선,1005,대방,6360,6476",

"20150101,경부선,1006,영등포,37247,36102",

"20150101,경원선,1008,이촌,1940,1507",

"20150101,경원선,1009,서빙고,911,1000",

"20150101,경원선,1010,한남,1885,1863",

"20150101,경원선,1011,옥수,43,37"]


In [4]:
_subRdd = spark.sparkContext.parallelize(_sub)

In [5]:
_subRdd.map(lambda x:x.split(',')).map(lambda x:int(x[4])).collect()

[6199,
 7982,
 17406,
 515,
 6879,
 20031,
 9519,
 7465,
 3943,
 4340,
 28980,
 23021,
 6360,
 37247,
 1940,
 911,
 1885,
 43]

In [6]:
_subLineByPassengers=_subRdd.map(lambda x:x.split(',')).map(lambda x: (x[1],int(x[4])))
sum_counts = _subLineByPassengers.combineByKey(
    (lambda x: (x, 1)), # the initial value, with value x and count 1
    (lambda acc, value: (acc[0]+value, acc[1]+1)), # how to combine a pair value with the accumulator: sum value, and increment count
    (lambda acc1, acc2: (acc1[0]+acc2[0], acc1[1]+acc2[1])) # combine accumulators
)

In [7]:
for i in sum_counts.collect():
    for each in i:
        print (each, end=" ")
    print()

2호선 (31587, 3) 
3호선 (36944, 4) 
4호선 (11408, 2) 
경부선 (99948, 5) 
경원선 (4779, 4) 


In [8]:
averageByKey = sum_counts.map(lambda x: (x[0],x[1][0]/x[1][1]))

In [9]:
averageByKey.collect()

[('2호선', 10529.0),
 ('3호선', 9236.0),
 ('4호선', 5704.0),
 ('경부선', 19989.6),
 ('경원선', 1194.75)]

In [10]:
_subRdd.map(lambda x:x.split(',')).map(lambda x:int(x[5])).collect()

[6219,
 8946,
 15241,
 538,
 6260,
 19470,
 11029,
 7574,
 10823,
 4535,
 27684,
 23862,
 6476,
 36102,
 1507,
 1000,
 1863,
 37]

In [11]:
_subLineByPassengers=_subRdd.map(lambda x:x.split(',')).map(lambda x: (x[1],int(x[5])))
sum_counts = _subLineByPassengers.combineByKey(
    (lambda x: (x, 1)), # the initial value, with value x and count 1
    (lambda acc, value: (acc[0]+value, acc[1]+1)), # how to combine a pair value with the accumulator: sum value, and increment count
    (lambda acc1, acc2: (acc1[0]+acc2[0], acc1[1]+acc2[1])) # combine accumulators
)

In [12]:
for i in sum_counts.collect():
    for each in i:
        print (each, end=" ")
    print()

2호선 (30406, 3) 
3호선 (37297, 4) 
4호선 (18397, 2) 
경부선 (98659, 5) 
경원선 (4407, 4) 


In [13]:
averageByKey = sum_counts.map(lambda x: (x[0],x[1][0]/x[1][1]))

In [14]:
averageByKey.collect()

[('2호선', 10135.333333333334),
 ('3호선', 9324.25),
 ('4호선', 9198.5),
 ('경부선', 19731.8),
 ('경원선', 1101.75)]